In [1]:
import httpx
import json

BASE_URL = "http://127.0.0.1:8000"
print("API Test Notebook ready ✅")

API Test Notebook ready ✅


In [2]:
response = httpx.get(f"{BASE_URL}/health")
print(json.dumps(response.json(), indent=2))

{
  "status": "healthy",
  "timestamp": "2026-04-24T20:26:45.146643",
  "models": {
    "claim_extractor": "groq/llama-3.3-70b-versatile",
    "span_retriever": "sentence-transformers/all-MiniLM-L6-v2",
    "nli_checker": "cross-encoder/nli-deberta-v3-base"
  }
}


In [3]:
response = httpx.get(f"{BASE_URL}/")
print(json.dumps(response.json(), indent=2))

{
  "name": "VeriFaith API",
  "version": "1.0.0",
  "status": "running",
  "docs": "/docs"
}


In [4]:
faithful_payload = {
    "answer": "The Eiffel Tower was built in 1889 and stands 330 metres tall. It was designed by Gustave Eiffel and is located in Paris, France.",
    "source_docs": [
        """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
        in Paris, France. It was constructed between 1887 and 1889 as the centerpiece 
        of the 1889 World's Fair. The tower was designed and built by Alexandre Gustave Eiffel, 
        a French civil engineer. It stands 330 metres tall and is one of the most recognizable 
        structures in the world."""
    ],
    "include_full_report": True
}

response = httpx.post(
    f"{BASE_URL}/evaluate",
    json=faithful_payload,
    timeout=120.0          # models need time to run
)

result = response.json()
print(f"Status Code     : {response.status_code}")
print(f"Faithfulness    : {result['faithfulness_score']}")
print(f"Verdict         : {result['verdict']}")
print(f"Supported       : {result['supported']}")
print(f"Contradicted    : {result['contradicted']}")

Status Code     : 200
Faithfulness    : 0.5
Verdict         : PARTIAL
Supported       : 2
Contradicted    : 0


In [5]:
hallucinated_payload = {
    "answer": "The Eiffel Tower was built in 1950 and stands 500 metres tall. It was designed by Leonardo da Vinci and is located in London.",
    "source_docs": [
        """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
        in Paris, France. It was constructed between 1887 and 1889 as the centerpiece 
        of the 1889 World's Fair. The tower was designed and built by Alexandre Gustave Eiffel, 
        a French civil engineer. It stands 330 metres tall and is one of the most recognizable 
        structures in the world."""
    ],
    "include_full_report": True
}

response = httpx.post(
    f"{BASE_URL}/evaluate",
    json=hallucinated_payload,
    timeout=120.0
)

result = response.json()
print(f"Status Code     : {response.status_code}")
print(f"Faithfulness    : {result['faithfulness_score']}")
print(f"Verdict         : {result['verdict']}")
print(f"Contradicted    : {result['contradicted']}")

if result["contradiction_report"]:
    print(f"\nContradictions Found:")
    for c in result["contradiction_report"]:
        print(f"  ❌ {c['claim']}")
        print(f"     Conflicts: {c['conflicts_with'][:70]}...")

Status Code     : 200
Faithfulness    : 0.0
Verdict         : UNFAITHFUL
Contradicted    : 4

Contradictions Found:
  ❌ The Eiffel Tower was built in 1950
     Conflicts: It was constructed between 1887 and 1889 as the centerpiece 
        o...
  ❌ The Eiffel Tower stands 500 metres tall
     Conflicts: It stands 330 metres tall and is one of the most recognizable 
       ...
  ❌ The Eiffel Tower was designed by Leonardo da Vinci
     Conflicts: The tower was designed and built by Alexandre Gustave Eiffel, 
       ...
  ❌ The Eiffel Tower is located in London
     Conflicts: The Eiffel Tower is a wrought-iron lattice tower located on the Champ ...


In [6]:
print(json.dumps(result, indent=2))

{
  "verifaith_version": "1.0.0",
  "evaluated_at": "2026-04-24T20:27:18.455531",
  "faithfulness_score": 0.0,
  "verdict": "UNFAITHFUL",
  "total_claims": 4,
  "supported": 0,
  "unverified": 0,
  "contradicted": 4,
  "has_contradictions": true,
  "claim_breakdown": [
    {
      "claim_id": 1,
      "claim_text": "The Eiffel Tower was built in 1950",
      "status": "contradicted",
      "confidence": 0.9998,
      "evidence_span": "It was constructed between 1887 and 1889 as the centerpiece \n        of the 1889 World's Fair.",
      "source_doc": 0,
      "was_reranked": true
    },
    {
      "claim_id": 2,
      "claim_text": "The Eiffel Tower stands 500 metres tall",
      "status": "contradicted",
      "confidence": 0.7144,
      "evidence_span": "It stands 330 metres tall and is one of the most recognizable \n        structures in the world.",
      "source_doc": 0,
      "was_reranked": true
    },
    {
      "claim_id": 3,
      "claim_text": "The Eiffel Tower was designe

In [7]:
print("""
VeriFaith API is also available via Swagger UI:

  http://127.0.0.1:8000/docs

Open this in your browser to:
  → See all endpoints
  → Test requests interactively
  → View request/response schemas
  → Share with teammates
""")


VeriFaith API is also available via Swagger UI:

  http://127.0.0.1:8000/docs

Open this in your browser to:
  → See all endpoints
  → Test requests interactively
  → View request/response schemas
  → Share with teammates

